In [7]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading stop: 100%|██████████| 1500/1500 [00:05<00:00, 251.20it/s]



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:01<00:00, 242.19it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


In [8]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu/'

# Ensure the Output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1 & 2: Optuna Optimization for LDA & Logistic Regression combined
print("\n--- Starting Optuna optimization with Focus on Targets ---")
def objective(trial):
    # --- LDA PARAMETERS ---
    lda_shrinkage = trial.suggest_float('lda_shrinkage', 0.0, 1.0)
    lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=lda_shrinkage, n_components=3)
    
    X_train_lda = lda.fit_transform(X_train, y_train)
    X_test_lda = lda.transform(X_test)
    
    # --- LOGISTIC REGRESSION PARAMETERS ---
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    # --- CLASS PRIORITIZATION ---
    target_weight = trial.suggest_float('target_weight', 1.0, 5.0)
    class_weights = {0: target_weight, 1: target_weight, 2: target_weight, 3: 1.0}
    
    lr = LogisticRegression(C=c_val, class_weight=class_weights, max_iter=5000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 3. TRAIN FINAL PIPELINE WITH BEST PARAMETERS
print("\n--- Training Final Lightweight & Focused Classifier ---")

best_shrinkage = study.best_params['lda_shrinkage']
best_C = study.best_params['C']
best_target_weight = study.best_params['target_weight']

# Train Final LDA (using 'eigen')
final_lda = LinearDiscriminantAnalysis(solver='eigen', shrinkage=best_shrinkage, n_components=3)
X_train_lda_final = final_lda.fit_transform(X_train, y_train)
X_test_lda_final = final_lda.transform(X_test)

# Train Final Logistic Regression
final_weights = {0: best_target_weight, 1: best_target_weight, 2: best_target_weight, 3: 1.0}
final_lr = LogisticRegression(C=best_C, class_weight=final_weights, max_iter=5000, random_state=42)
final_lr.fit(X_train_lda_final, y_train)

# Evaluate
preds = final_lr.predict(X_test_lda_final)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 4. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters ---")

# Extract LDA parameters (No xbar_ needed for 'eigen' solver!)
lda_scalings = final_lda.scalings_.tolist() 

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           
lr_intercept = final_lr.intercept_.tolist() 

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Robust LDA (Eigen) + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

[I 2026-06-30 17:05:46,927] A new study created in memory with name: no-name-9924bcbe-9e70-4a07-ac12-fbe8adf6f237
[I 2026-06-30 17:05:46,975] Trial 0 finished with value: 0.7039199332777314 and parameters: {'lda_shrinkage': 0.47906432007021094, 'C': 44.08664245096021, 'target_weight': 3.8265640115341504}. Best is trial 0 with value: 0.7039199332777314.
[I 2026-06-30 17:05:47,004] Trial 1 finished with value: 0.7331109257714762 and parameters: {'lda_shrinkage': 0.0927548927013212, 'C': 0.002339249309100525, 'target_weight': 2.746161859700628}. Best is trial 1 with value: 0.7331109257714762.
[I 2026-06-30 17:05:47,038] Trial 2 finished with value: 0.7406171809841534 and parameters: {'lda_shrinkage': 0.09025244141999567, 'C': 11.906173895702436, 'target_weight': 3.85149233602711}. Best is trial 2 with value: 0.7406171809841534.
[I 2026-06-30 17:05:47,073] Trial 3 finished with value: 0.7456213511259383 and parameters: {'lda_shrinkage': 0.3680947023829455, 'C': 0.006017190370704378, 'targe

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

--- Starting Optuna optimization with Focus on Targets ---


[I 2026-06-30 17:05:47,139] Trial 5 finished with value: 0.6113427856547122 and parameters: {'lda_shrinkage': 0.584782614240283, 'C': 0.00020137408051719134, 'target_weight': 3.0489652337388344}. Best is trial 3 with value: 0.7456213511259383.
[I 2026-06-30 17:05:47,181] Trial 6 finished with value: 0.7030859049207673 and parameters: {'lda_shrinkage': 0.5313825785307097, 'C': 0.8882136565116939, 'target_weight': 3.6347944521864632}. Best is trial 3 with value: 0.7456213511259383.
[I 2026-06-30 17:05:47,218] Trial 7 finished with value: 0.5929941618015012 and parameters: {'lda_shrinkage': 0.7746825395032881, 'C': 0.00015914648805361812, 'target_weight': 4.137177911864812}. Best is trial 3 with value: 0.7456213511259383.
[I 2026-06-30 17:05:47,257] Trial 8 finished with value: 0.7397831526271893 and parameters: {'lda_shrinkage': 0.38476873415200663, 'C': 78.14004575319193, 'target_weight': 1.8893391138277642}. Best is trial 3 with value: 0.7456213511259383.
[I 2026-06-30 17:05:47,324] Tr


Best Optuna Parameters: {'lda_shrinkage': 0.053834383532462136, 'C': 0.01147434336359507, 'target_weight': 1.0075660285777839}
Best Accuracy during CV: 79.23%

--- Training Final Lightweight & Focused Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.82      0.82      0.82       299
         off       0.80      0.80      0.80       300
        stop       0.88      0.78      0.83       300
     unknown       0.69      0.76      0.72       300

    accuracy                           0.79      1199
   macro avg       0.80      0.79      0.79      1199
weighted avg       0.80      0.79      0.79      1199


--- Exporting LDA and Classifier Parameters ---
Pipeline successfully exported to: ../edge_mcu/model_data_lr.py


In [9]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Super-optimized Inference Engine for MicroPython (LDA Eigen + Logistic Regression).
    File size: < 3KB | Execution time: < 2ms
    """
    num_original_features = 39 # 13 MFCCs * 3 Time Frames
    num_super_features = 3     # Reduced dimensions by LDA
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation (Direct Matrix Multiplication) ---
    # Since we used 'eigen' solver, we directly multiply raw features by SCALINGS
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Start with the bias/intercept for this class
        score = model_data_lr.LR_INTERCEPT[0][c]
        
        # Dot product: Super-features * Class Weights
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_classes):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # 39 features extracted from mic (VAD applied)
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data_lr.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data_lr.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: off and actuall is: 2
Predicted Word: stop and actuall is: 3
Predicted Word: off and actuall is: 2
Predicted Word: stop and actuall is: 2
Predicted Word: off and actuall is: 4
Predicted Word: unknown and actuall is: 4
Predicted Word: off and actuall is: 2
Predicted Word: stop and actuall is: 3
Predicted Word: off and actuall is: 4
Predicted Word: stop and actuall is: 3
